In [11]:
from pprint import pprint

import mlflow 
import pandas as pd 

from pytorch_pipeline.utils import resolve_uri
from pytorch_pipeline.train.factory import set_device


In [20]:
pd.options.display.float_format = '{:.9f}'.format
pd.set_option('display.max_rows', None)


In [5]:
model_name = 'cv_pheno_bioclip2'
model_version = 4

In [6]:
mlflow.set_tracking_uri(resolve_uri())
device = set_device('cuda')

# Construct the model URI
model_uri = f"models:/{model_name}/{model_version}"

# Load the model
model = mlflow.pyfunc.load_model(model_uri)


Running on cuda


{'class_thresholds': '/tmp/tmp1l3bsiwl/artifacts/class_thresholds.json', 'model_params': '/tmp/tmp1l3bsiwl/artifacts/model_params.json', 'model_state_dict': '/tmp/tmp1l3bsiwl/artifacts/model_state_dict.pt'}
{'Flowering': 0.4099999999999999, 'Fruiting': 0.7599999999999999, 'Flower_Budding': 0.6499999999999999}


In [7]:
pprint(model._model_impl.python_model.model_params)
pprint(model._model_impl.python_model.class_thresholds)

ModelParams(backbone='bioclip2',
            head_neurons=256,
            head_outputs=1,
            head_dropout_prob=0.5,
            attention_neurons=128,
            attention_dropout_prob=0.0,
            start_unfreezed=1,
            gated=True)
array([0.41, 0.76, 0.65])


In [8]:
paths = []
imgs =[
    [161284642,161284591],
    [147565223]
]
for b in imgs:
    paths.append([f"/home/etienne/projects/inat-phenology-cv/data/images/{i}.jpg" for i in b])

pprint(paths)

[['/home/etienne/projects/inat-phenology-cv/data/images/161284642.jpg',
  '/home/etienne/projects/inat-phenology-cv/data/images/161284591.jpg'],
 ['/home/etienne/projects/inat-phenology-cv/data/images/147565223.jpg']]


In [21]:
model_input = pd.DataFrame(
    {
        "observation_id" : [97028130, 89365289],
        "paths" : paths
    }
)
display(model_input)

,observation_id,paths
0,97028130,[/home/etienne/projects/inat-phenology-cv/data...
1,89365289,[/home/etienne/projects/inat-phenology-cv/data...


In [14]:
predictions, weights = model.predict(model_input)



In [17]:
for p, w in zip(predictions, weights):
    print(p)
    pprint(w)
    print("\n")
    

[1 1 0]
{'Flower_Budding': [0.07284402847290039, 0.9271559715270996],
 'Flowering': [0.15610887110233307, 0.8438911437988281],
 'Fruiting': [0.49824532866477966, 0.501754641532898]}


[1 0 1]
{'Flower_Budding': [1.0], 'Flowering': [1.0], 'Fruiting': [1.0]}


